# Module 3 — Model Training and Baseline Comparison

This notebook trains a LightGBM regressor to predict per-game fantasy points and evaluates it against a rule-of-thumb baseline (the player's trailing 10-game average). The split is strictly chronological: 2021–22 through 2023–24 for training, 2024–25 for testing. A random split would permit the model to train on future games and predict past ones, which inflates test metrics while masking a model that will fail on live data.

Metrics are reported alongside the baseline in every case. A model that cannot improve on a formula computable in Excel does not earn its place in a production pipeline, and reporting metrics in isolation is the easiest way to look more useful than one actually is.

**Result:** LightGBM MAE 10.09 FP vs. baseline MAE 10.19 FP — a one-percent improvement. The full interpretation of this result (why the gap is small, and what it implies about feature choice) is written below alongside the output.

## Setup — load the feature-engineered data

In [18]:
import pandas as pd
import numpy as np

df = pd.read_parquet('data/processed/model_df.parquet')

print(f'Shape: {df.shape}')
print(f'Columns: {df.columns.tolist()}')
print(f'Date range: {df["GAME_DATE"].min()} to {df["GAME_DATE"].max()}')
df.head()

Shape: (104641, 15)
Columns: ['PLAYER_ID', 'PLAYER_NAME', 'GAME_DATE', 'TEAM_ABBREVIATION', 'OPP', 'rest_days_capped', 'returned_from_absence', 'rolling_fp_5', 'rolling_fp_10', 'is_home', 'opp_fp_allowed', 'rolling_min_10', 'rolling_fp_std_10', 'fp_momentum', 'FP']
Date range: 2021-10-19 00:00:00 to 2025-04-13 00:00:00


,PLAYER_ID,PLAYER_NAME,GAME_DATE,TEAM_ABBREVIATION,OPP,rest_days_capped,returned_from_absence,rolling_fp_5,rolling_fp_10,is_home,opp_fp_allowed,rolling_min_10,rolling_fp_std_10,fp_momentum,FP
0,2207,Joe Johnson,2021-12-22,BOS,CLE,NaN,False,NaN,NaN,1,191.9,NaN,NaN,NaN,3
1,2544,LeBron James,2021-10-19,LAL,GSW,NaN,False,NaN,NaN,1,NaN,NaN,NaN,NaN,60
2,2544,LeBron James,2021-10-22,LAL,PHX,3.0,False,NaN,NaN,1,NaN,NaN,NaN,NaN,38
3,2544,LeBron James,2021-10-24,LAL,MEM,2.0,False,NaN,NaN,1,NaN,NaN,NaN,NaN,45
4,2544,LeBron James,2021-10-29,LAL,CLE,5.0,False,NaN,NaN,1,NaN,NaN,NaN,NaN,46


## Chronological train/test split

- **Train:** everything BEFORE 2024-10-01 (3 seasons of history)
- **Test:** 2024-25 season onwards (what we 'predict')

First drop rows where features are NaN (early-career games can't be predicted — `rolling_fp_10` needs 10 games of history).

In [19]:
# Drop rows with any NaN features (early-career games can't be predicted yet)
feature_cols = ['rest_days_capped', 'returned_from_absence',
                'rolling_fp_5', 'rolling_fp_10',
                'is_home', 'opp_fp_allowed', 'rolling_min_10',
                'rolling_fp_std_10', 'fp_momentum']

clean_df = df.dropna(subset=feature_cols).copy()
print(f'After dropping NaN features: {clean_df.shape[0]:,} rows (was {df.shape[0]:,})')

# Chronological split
train_mask = clean_df['GAME_DATE'] < '2024-10-01'
train = clean_df[train_mask].copy()
test  = clean_df[~train_mask].copy()

print(f'\nTrain: {train.shape[0]:,} rows  ({train["GAME_DATE"].min().date()} to {train["GAME_DATE"].max().date()})')
print(f'Test:  {test.shape[0]:,} rows  ({test["GAME_DATE"].min().date()} to {test["GAME_DATE"].max().date()})')

After dropping NaN features: 96,175 rows (was 104,641)

Train: 70,843 rows  (2021-11-08 to 2024-04-14)
Test:  25,332 rows  (2024-10-22 to 2025-04-13)


In [20]:
# Baseline prediction: just guess rolling_fp_10
test['baseline_pred'] = test['rolling_fp_10']

# How wrong were we? MAE = Mean Absolute Error
baseline_mae = (test['FP'] - test['baseline_pred']).abs().mean()

print(f'Baseline MAE: {baseline_mae:.2f} FP')
print(f'(interpretation: on average, our dumb guess is off by {baseline_mae:.2f} fantasy points)')

Baseline MAE: 10.19 FP
(interpretation: on average, our dumb guess is off by 10.19 fantasy points)


## Train LightGBM

Standard LightGBM regressor with sensible defaults.

**X = features matrix. y = target vector.** Convention everywhere in ML.

In [21]:
import lightgbm as lgb
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Split features (X) and target (y)
X_train = train[feature_cols]
y_train = train['FP']
X_test  = test[feature_cols]
y_test  = test['FP']

# Train LightGBM with sensible defaults
model = lgb.LGBMRegressor(
    n_estimators=500,      # up to 500 trees
    learning_rate=0.05,    # how much each tree corrects (small = cautious)
    num_leaves=31,         # tree complexity
    random_state=42,       # reproducibility
    verbose=-1,            # quiet training
)

model.fit(X_train, y_train)
print('Training complete.')

Training complete.


In [22]:
# Predict on test set and evaluate
preds = model.predict(X_test)

mae  = mean_absolute_error(y_test, preds)
rmse = mean_squared_error(y_test, preds) ** 0.5
r2   = r2_score(y_test, preds)

print(f'LightGBM MAE:  {mae:.2f}  (baseline was {baseline_mae:.2f})')
print(f'LightGBM RMSE: {rmse:.2f}')
print(f'LightGBM R2:   {r2:.3f}')
print()
improvement = baseline_mae - mae
print(f'Improvement over baseline: {improvement:.2f} FP ({100*improvement/baseline_mae:.1f}%)')

LightGBM MAE:  10.09  (baseline was 10.19)
LightGBM RMSE: 12.86
LightGBM R2:   0.426

Improvement over baseline: 0.10 FP (1.0%)


In [23]:
# Which features mattered most? LightGBM gives us this for free.
importance = pd.DataFrame({
    'feature': feature_cols,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

print('Feature importance (higher = more useful to the model):')
print(importance.to_string(index=False))

Feature importance (higher = more useful to the model):
              feature  importance
       opp_fp_allowed        2738
       rolling_min_10        2575
    rolling_fp_std_10        2407
         rolling_fp_5        2158
          fp_momentum        2124
        rolling_fp_10        1946
     rest_days_capped         637
              is_home         345
returned_from_absence          70
